# Boutique Hotel Operations Assistant

## Set Up Local LLM and Embedding Model

In [30]:
from llama_index.core import Settings
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.llms.ollama import Ollama

# 1. Local Embeddings Model (Ollama)
Settings.embed_model = OllamaEmbedding(
    model_name="nomic-embed-text"
)

# 2. Local LLM via Ollama (Qwen2.5:3b)
Settings.llm = Ollama(
    model="qwen2.5:3b", request_timeout=120.0, base_url="http://localhost:11434"
)

## Single Query Engine to handle each type of Data

In [31]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
documents = SimpleDirectoryReader(input_dir="./data").load_data()

# 3. Create a single Vector Store Index over everything
index = VectorStoreIndex.from_documents(documents)

# 4. Create your query engine
query_engine = index.as_query_engine()

# Query across all files instantly!
response = query_engine.query("What is the name of our Hotel")
print(response)

2026-07-22 23:23:29,985 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-07-22 23:23:30,142 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-07-22 23:23:30,164 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


2026-07-22 23:23:33,908 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
GRAND HERITAGE HOTEL


## Combine into a Multi-Tool Agent / Router Engine

In [32]:
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.experimental.query_engine import PandasQueryEngine



In [33]:
# ---------------------------------------------------------------------------
# 1. Tool 1 — Guest Policies & SOPs (VectorStoreIndex)
# ---------------------------------------------------------------------------
def build_policy_tool() -> QueryEngineTool:
    policy_docs = [
        documents(
            text=(
                "Check-in time is 3:00 PM and check-out time is 11:00 AM. "
                "Early check-in and late check-out are subject to availability "
                "and may incur a fee of $25 per hour."
            ),
            metadata={"section": "check-in-out"},
        ), # type: ignore
        documents(
            text=(
                "Cancellations made more than 48 hours before arrival are fully "
                "refundable. Cancellations within 48 hours are charged one night's "
                "stay. No-shows are charged the full reservation amount."
            ),
            metadata={"section": "cancellation-policy"},
        ), # type: ignore
        documents(
            text=(
                "Pets are allowed in designated pet-friendly rooms only, with a "
                "non-refundable cleaning fee of $75 per stay. Maximum of two pets "
                "under 50 lbs each."
            ),
            metadata={"section": "pet-policy"},
        ), # type: ignore
        documents(
            text=(
                "In the event of a fire alarm, front desk staff must immediately "
                "notify the fire department, activate the evacuation announcement, "
                "and direct guests to the nearest stairwell. Elevators must not be used."
            ),
            metadata={"section": "emergency-sop"},
        ),# type: ignore
        documents(
            text=(
                "Guest complaints regarding room quality should be resolved within "
                "30 minutes when possible. Staff are authorized to offer a complimentary "
                "amenity or partial refund up to $50 without manager approval."
            ),
            metadata={"section": "complaint-resolution-sop"},
        ),# type: ignore
    ]
 
    index = VectorStoreIndex.from_documents(policy_docs)
    query_engine = index.as_query_engine(similarity_top_k=3)
 
    return QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name="guest_policies_sops",
        description=(
            "Answers questions about guest-facing policies and internal SOPs: "
            "check-in/check-out rules, cancellation policy, pet policy, "
            "emergency procedures, and complaint-handling procedures. "
            "Use for 'what is our policy on...' or 'what should staff do if...' questions."
        ),
    )

In [34]:
import pandas as pd
# ---------------------------------------------------------------------------
# 2. Tool 2 — Bookings & Revenue (Pandas Engine)
# ---------------------------------------------------------------------------
def build_bookings_tool() -> QueryEngineTool:
    bookings_df = pd.DataFrame(
        {
            "booking_id": [f"BK{i:04d}" for i in range(1, 11)],
            "room_type": [
                "Standard", "Standard", "Deluxe", "Suite", "Standard",
                "Deluxe", "Suite", "Standard", "Deluxe", "Suite",
            ],
            "check_in": pd.to_datetime(
                [
                    "2026-07-01", "2026-07-02", "2026-07-02", "2026-07-03",
                    "2026-07-04", "2026-07-05", "2026-07-05", "2026-07-06",
                    "2026-07-07", "2026-07-08",
                ]
            ),
            "nights": [2, 1, 3, 5, 2, 4, 2, 1, 3, 6],
            "rate_per_night": [120, 120, 180, 320, 120, 180, 320, 120, 180, 320],
            "status": [
                "checked_out", "checked_out", "in_house", "confirmed",
                "checked_out", "in_house", "confirmed", "cancelled",
                "in_house", "confirmed",
            ],
        }
    )
    bookings_df["total_revenue"] = bookings_df["nights"] * bookings_df["rate_per_night"]
 
    query_engine = PandasQueryEngine(df=bookings_df, verbose=True) # type: ignore
 
    return QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name="bookings_revenue",
        description=(
            "Answers quantitative questions about bookings, occupancy, room types, "
            "nights stayed, rates, and revenue using a pandas DataFrame of reservation "
            "records. Use for 'how much revenue', 'how many bookings', 'average rate', "
            "'occupancy by room type' type questions."
        ),
    )
 

In [35]:
# ---------------------------------------------------------------------------
# 3. Tool 3 — Housekeeping & Maintenance (Pandas Engine)
# ---------------------------------------------------------------------------
def build_housekeeping_tool() -> QueryEngineTool:
    hk_df = pd.DataFrame(
        {
            "ticket_id": [f"HK{i:04d}" for i in range(1, 9)],
            "room_number": [101, 102, 205, 210, 305, 310, 402, 405],
            "issue_type": [
                "cleaning", "cleaning", "maintenance", "cleaning",
                "maintenance", "maintenance", "cleaning", "maintenance",
            ],
            "description": [
                "Standard turnover clean",
                "Deep clean requested by guest",
                "AC unit not cooling",
                "Standard turnover clean",
                "Leaking bathroom faucet",
                "TV remote not working",
                "Standard turnover clean",
                "Broken window latch",
            ],
            "priority": ["low", "medium", "high", "low", "high", "low", "low", "medium"],
            "status": [
                "completed", "completed", "open", "completed",
                "in_progress", "open", "completed", "open",
            ],
            "assigned_to": [
                "Team A", "Team A", "Maintenance", "Team B",
                "Maintenance", "Maintenance", "Team B", "Maintenance",
            ],
        }
    )
 
    query_engine = PandasQueryEngine(df=hk_df, verbose=True) # type: ignore
 
    return QueryEngineTool.from_defaults(
        query_engine=query_engine,
        name="housekeeping_maintenance",
        description=(
            "Answers questions about housekeeping and maintenance tickets: open issues, "
            "priority levels, room-specific problems, and team assignments using a pandas "
            "DataFrame of ticket records. Use for 'what rooms need attention', "
            "'how many open tickets', 'who is assigned to' type questions."
        ),
    )
 

In [36]:
# ---------------------------------------------------------------------------
# 4. Assemble the Router
# ---------------------------------------------------------------------------
def build_router() -> RouterQueryEngine:
    tools = [
        build_policy_tool(),
        build_bookings_tool(),
        build_housekeeping_tool(),
    ]
 
    router = RouterQueryEngine(
        selector=LLMSingleSelector.from_defaults(),
        query_engine_tools=tools,
        verbose=True,
    )
    return router
 

## Test Sample Agent Queries

In [37]:
# Query 1: Unstructured Policy Retrieval (from .txt / .pdf)
response = query_engine.query(
    "What is our policy for early check-in, and is the fee waived for VIP members?"
)
print(response)

2026-07-22 23:23:34,184 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-07-22 23:23:37,093 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Our hotel has a policy where early check-in can be done before 1:00 PM, but this requires availability and incurs an early arrival fee of $45. For Gold and Platinum tier VIP members, the early check-in fee is waived.


In [38]:
# Query 2: Operational Procedure (from .pdf)
response = query_engine.query(
    "How should staff handle a guest complaint about a noisy room using the H.E.A.T. protocol?"
)
print(response)

2026-07-22 23:23:37,264 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-07-22 23:23:45,172 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
When handling a guest complaint about a noisy AC unit during their stay at the Grand Heritage Hotel, front desk staff should follow the H.E.A.T. protocol:

1. **Hear**: Listen attentively to the guest's concerns without interrupting or making excuses.
2. **Empathize**: Acknowledge the guest's frustration directly, such as "I completely understand how inconvenient a noisy AC unit is during your stay."
3. **Apologize**: Represent hotel standards and give a sincere apology, for example, "We truly apologize for any inconvenience this has caused you."
4. **Take Action**: Offer a solution immediately to address the noise issue. Possible solutions could include arranging for a room change or providing a complimentary breakfast pass.


In [39]:
# Query 3: Structured Data Query (from .xlsx)
response = query_engine.query(
    "Which rooms currently have open maintenance tickets and need HVAC repairs?"
)
print(response)

2026-07-22 23:23:45,347 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-07-22 23:23:49,757 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Room No: Room 102 has an open maintenance ticket with a reported issue of Bathroom sink drain slow, which falls under the category of Plumbing. Since it is categorized as a High priority, it does not pertain to HVAC repair needs.

No room currently listed in the provided data has an open maintenance ticket specifically for HVAC repairs (Air Conditioning cooling weak). Therefore, no rooms need immediate HVAC repairs based on the given information.


In [40]:
# Query 4: Multi-document Combined Query
response = query_engine.query(
    "Check the status of Suite 401: Are there any open maintenance tickets, and what are the VIP preparation requirements for it?"
)
print(response)

2026-07-22 23:23:49,894 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


2026-07-22 23:23:57,637 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
To check the status of Suite 401 at Grand Heritage Boutique Hotel regarding open maintenance tickets, you would need to refer to the Executive Dashboard in your data file. This dashboard provides a live overview of all operations including room statuses.

As for the VIP preparation requirements for Suite 401, since it is an Ocean View Deluxe King/Twin suite, we follow these specific guidelines:
- The room category requires a cleaning every two days.
- Key amenities such as luxury robes, heritage toiletries set, and two bottled waters are essential.
- An additional full refresh including espresso pods, fresh flowers, and a welcome fruit basket should be provided.

The H.E.A.T. Method protocol must also be followed for any guest issues or complaints related to this suite.
